In [2]:
"""
Phenyo Thato Molete- 216038155

tests.py
Assignment 6 test suite: in alignment with the smoke tests in the Lecture 6 notebook
(_run_a6_smoke) so the official checks are covered.

Run:
    python tests.py
"""

from mining import Block, Blockchain, GENESIS_PREV, mine_block, make_coinbase, mine_and_append, difficulty_study


def test_mine_block_meets_difficulty():
    blk = Block(1, [{"x": 1}], GENESIS_PREV)
    mined, attempts, secs = mine_block(blk, difficulty=2)
    assert mined.hash.startswith("00")
    assert mined.hash == mined.compute_hash()
    assert attempts >= 1
    assert secs >= 0.0
    print("test_mine_block_meets_difficulty: PASS")


def test_coinbase_shape():
    tx = make_coinbase("MinerPhenyo", reward=50.0)
    assert tx["sender"] == "NETWORK"
    assert tx["recipient"] == "MinerPhenyo"
    assert tx["amount"] == 50.0
    assert "timestamp" in tx
    assert tx["type"] == "coinbase"
    print("test_coinbase_shape: PASS")


def test_mine_and_append_verifies():
    chain = Blockchain()
    mined, attempts, secs = mine_and_append(
        chain,
        [{"sender": "Phenyo", "recipient": "Thato", "amount": 1}],
        miner_address="MinerPhenyo",
        difficulty=2,
    )
    assert mined.hash.startswith("00")
    assert mined.transactions[0]["sender"] == "NETWORK"
    assert mined.transactions[0]["recipient"] == "MinerPhenyo"
    assert chain.verify_chain(difficulty=2)
    assert attempts >= 1
    assert secs >= 0.0
    print("test_mine_and_append_verifies: PASS")


def test_difficulty_study_three_levels():
    study_rows = difficulty_study((2, 3))
    assert len(study_rows) == 2
    for row, d in zip(study_rows, (2, 3)):
        assert row["difficulty"] == d
        assert row["hash"].startswith("0" * d)
        assert "attempts" in row
        assert "seconds" in row
    print("test_difficulty_study_three_levels: PASS")


def test_append_rejects_unmet_difficulty():
    bare = Blockchain()
    unmined = Block(1, [{"x": 1}], bare.tip().hash)
    try:
        bare.append_block(unmined, difficulty=2)
        raised = False
    except ValueError:
        raised = True
    assert raised is True
    print("test_append_rejects_unmet_difficulty: PASS")


def test_tampering_after_mining_fails_verify():
    chain = Blockchain()
    mined, _, _ = mine_and_append(
        chain,
        [{"sender": "Phenyo", "recipient": "Thato", "amount": 1}],
        miner_address="MinerPhenyo",
        difficulty=2,
    )
    mined.transactions[1]["amount"] = 999  # mutate after PoW; stored hash goes stale
    assert chain.verify_chain(difficulty=2) is False
    print("test_tampering_after_mining_fails_verify: PASS")


def test_coinbase_placed_before_mempool_txs():
    chain = Blockchain()
    mined, _, _ = mine_and_append(
        chain,
        [{"sender": "Phenyo", "recipient": "Thato", "amount": 1}],
        miner_address="MinerPhenyo",
        difficulty=2,
    )
    assert mined.transactions[0]["type"] == "coinbase"
    assert mined.transactions[1]["sender"] == "Phenyo"
    print("test_coinbase_placed_before_mempool_txs: PASS")


def run_all():
    tests = [
        test_mine_block_meets_difficulty,
        test_coinbase_shape,
        test_mine_and_append_verifies,
        test_difficulty_study_three_levels,
        test_append_rejects_unmet_difficulty,
        test_tampering_after_mining_fails_verify,
        test_coinbase_placed_before_mempool_txs,
    ]
    for t in tests:
        t()
    print(f"\nAll {len(tests)} tests passed.")


if __name__ == "__main__":
    run_all()

ModuleNotFoundError: No module named 'mining'